# 09 — Definitional Patterns: what do LLMs encode about "Digital Humanities"?

The earlier notebooks (02–08) examine *what the models produce* (translations, flags, embeddings, search results). This notebook turns to a different question: **what do the models think Digital Humanities *is*?** The rationale column gives us an indirect window into each model's internal definition of the term — and the variation across models, variants, and target languages reveals something about how thinly or richly each model has internalised a concept that DH scholars themselves struggle to define.

Four motivating questions structure the analysis:

1. **Definitional registers.** What *types* of definition do LLMs reach for? Scholarly/methodological framings (interdisciplinary, computational research), compositional/literal framings (digital + humanities), human-centered framings (people, culture, experience), or generic-knowledge framings (digital + learning/knowledge)?
2. **Frontier vs. smaller-model split.** Earlier findings (xmn, Himalayan template, Llama Indus Valley template) showed that smaller models exhibit *structural* failures of multilingual representation. Do they also exhibit *conceptual* thinness — relying on the literal compositional register where frontier models reach for scholarly framings?
3. **Component asymmetry.** Which side of *Digital Humanities* is harder for the models to encode — "digital" (concrete technical concept) or "humanities" (contested scholarly field)?
4. **Definitional richness ↔ translation success.** Do rationales that reach for richer scholarly framings correlate with translations that pass the quality-flag suite, or is there no relationship?

**Method**: keyword-based first pass for definitional registers; component-segmented pattern matching for the digital-vs-humanities asymmetry; cross-reference with `quality_flags.csv` for the richness-vs-success question. The keyword approach is necessarily coarse — a future-work section flags LLM-as-judge validation on a stratified sample as the natural next step.

**Input**: per-service rationale columns in `prompt_services/*_translations.csv` files.  
**Scope**: English-register variants only — `minimal`, `github_searcher`, `judge`. The `fluent_speaker` variant produces target-language rationales and is excluded from this analysis.

In [1]:
import os, sys, re
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import altair as alt

alt.data_transformers.enable('vegafusion')

sys.path.insert(0, str(Path('..').resolve()))
from scripts.utils import get_data_directory_path, get_language_family

DATA_DIR  = get_data_directory_path()
TERM      = 'Digital Humanities'
TERM_SLUG = TERM.lower().replace(' ', '_')

# English-rationale variants only — fluent_speaker uses target language
ENGLISH_VARIANTS = ['minimal', 'github_searcher', 'judge']
ALL_VARIANTS     = ['minimal', 'fluent_speaker', 'github_searcher', 'judge']

LLM_SERVICES = ['claude', 'openai', 'gemini', 'deepseek', 'llama', 'gemma', 'qwen', 'mistral']
FRONTIER     = {'claude', 'openai', 'gemini', 'deepseek'}
SMALL        = {'llama', 'gemma', 'qwen', 'mistral'}

PROMPT_DIR = os.path.join(DATA_DIR, 'translated_terms', TERM_SLUG, 'prompt_services')
print(f'Prompt service dir: {PROMPT_DIR}')

Retrieving translation pipeline data directory path...

Prompt service dir: /Users/zleblanc/CodingDH/translation_transmogrification_pipeline/datasets/translated_terms/digital_humanities/prompt_services


## 9.1 — Loading the rationale corpus

Build a single long-format DataFrame with one row per (service, variant, language) rationale. Drop empty/placeholder rationales (`'No rationale provided'`, `'N/A'`, etc.) and restrict to English-register variants. The output is the corpus that all subsequent analyses operate on.

In [2]:
_PLACEHOLDER_RAT = {
    'no rationale provided', 'no rationale', 'n/a', 'none',
    'not applicable', 'no explanation provided', 'no reason provided',
}

def _valid_rat(s):
    if not isinstance(s, str) or not s.strip(): return False
    return s.strip().rstrip('.').lower() not in _PLACEHOLDER_RAT

rows = []
for svc in LLM_SERVICES:
    for variant in ENGLISH_VARIANTS:
        path = os.path.join(PROMPT_DIR, f'{svc}_{variant}_translations.csv')
        if not os.path.exists(path): continue
        df = pd.read_csv(path, dtype=str)
        rat_col = f'{svc}_translation_rationale'
        if rat_col not in df.columns: continue
        for _, row in df.iterrows():
            rat = row.get(rat_col)
            if not _valid_rat(rat): continue
            rows.append({
                'service': svc,
                'variant': variant,
                'language_code': row.get('language_code'),
                'rationale': rat.strip(),
                'model_size': 'frontier' if svc in FRONTIER else 'small',
            })
rationale_df = pd.DataFrame(rows)
print(f'Total English-register rationales: {len(rationale_df)}')
print(f'  By service:\n{rationale_df["service"].value_counts().to_string()}')

Total English-register rationales: 20657
  By service:
service
claude      2640
qwen        2632
deepseek    2623
gemma       2613
mistral     2584
openai      2560
gemini      2558
llama       2447


## 9.2 — Definitional registers (keyword pass)

Four candidate "registers" of DH definition, each operationalised as a small set of phrasal patterns. A rationale may match multiple registers (definitions overlap in practice; this isn't an exclusive classification).

- **Compositional / literal**: rationale describes DH as a combination of *digital* and *humanities* without further scholarly framing. The default fallback register.
- **Scholarly / methodological**: rationale invokes interdisciplinarity, computational methods, scholarship, academic discipline — recognising DH as a *field with internal methodological commitments*.
- **Human-centered**: rationale appeals to humans, people, culture, or human experience — emphasising the *humanities subject* of DH.
- **Generic knowledge**: rationale glosses humanities as *knowledge*, *education*, *learning* — flattening the scholarly specificity into broader cognitive terms.

The contrast that matters most for the article is whether the model treats *humanities* as **a scholarly field** (rich), **a category of human experience** (medium), or **a generic knowledge domain** (thin). The compositional register is the baseline — present in nearly every rationale — so the comparative measure is which *non-compositional* register a model reaches for.

In [3]:
REGISTER_PATTERNS = {
    'compositional_literal': [
        r'\bcombines?\b.*\bdigital\b',
        r'\bdigital\b.*\b(?:and|with|plus)\b.*\bhumanities\b',
    ],
    'scholarly_methodological': [
        r'\binterdisciplinary\b',
        r'\bcomputational\s+method',
        r'\bdigital\s+(?:methods?|tools?)\s+(?:to|in|for)\s+(?:study|stud|research|analy)',
        r'\bscholarly\b.*\bdigital\b',
        r'\bdigital\b.*\bscholarship\b',
        r'\bacademic\s+(?:field|discipline)',
    ],
    'human_centered': [
        r'\b(?:study|exploration|investigation)\s+of\s+human\s+culture',
        r'\bhuman\s+experience\b',
        r'\bpeople\b.*\bdigital\b',
        r'\bhuman\b.*\b(?:digital|computer)\b.*\b(?:cultur|stud|experience)',
    ],
    'knowledge_generic': [
        r'\bdigital\b.*\bknowledge\b',
        r'\bknowledge\b.*\bdigital\b',
        r'\bdigital\s+(?:education|learning|wisdom)',
    ],
}

def matches_register(rationale, register):
    return any(re.search(p, rationale, re.IGNORECASE) for p in REGISTER_PATTERNS[register])

# Apply all four registers
for register in REGISTER_PATTERNS:
    rationale_df[f'reg_{register}'] = rationale_df['rationale'].apply(
        lambda r: matches_register(r, register)
    )

# Service-level summary
register_summary = (
    rationale_df.groupby('service')[[f'reg_{r}' for r in REGISTER_PATTERNS]]
    .mean()
    .mul(100).round(1)
    .rename(columns={f'reg_{r}': r for r in REGISTER_PATTERNS})
)
print('Definitional register prevalence (% of rationales per service):\n')
print(register_summary.to_string())

Definitional register prevalence (% of rationales per service):

          compositional_literal  scholarly_methodological  human_centered  knowledge_generic
service                                                                                     
claude                     65.2                      34.8            15.0               34.5
deepseek                   65.6                      39.9            17.9               19.5
gemini                     69.7                      46.7            23.1               27.3
gemma                      64.5                      22.5             7.0               31.6
llama                      66.5                       6.9             8.6               10.4
mistral                    55.1                       9.4             7.0                3.9
openai                     72.8                      38.1            23.3                8.4
qwen                       56.3                      10.9             2.9                9.4


In [4]:
# Heatmap — service × register, ordered by frontier-then-small
svc_order = sorted(rationale_df['service'].unique(),
                   key=lambda s: (s not in FRONTIER, s))
reg_order = ['compositional_literal', 'scholarly_methodological', 'human_centered', 'knowledge_generic']

long = (register_summary.reset_index()
        .melt(id_vars='service', var_name='register', value_name='pct'))

heat = alt.Chart(long).mark_rect(stroke='white', strokeWidth=2).encode(
    x=alt.X('register:N', sort=reg_order, title=None,
            axis=alt.Axis(labelAngle=-30)),
    y=alt.Y('service:N', sort=svc_order, title=None),
    color=alt.Color('pct:Q', scale=alt.Scale(scheme='blues', domain=[0, 100]),
                    title='% of rationales'),
    tooltip=['service:N','register:N','pct:Q'],
).properties(width=380, height=240,
             title=alt.Title('Definitional register × service',
                             subtitle='Frontier models (top 4) tend toward scholarly framing; smaller models stay compositional'))
txt = heat.mark_text(fontSize=10).encode(
    text=alt.Text('pct:Q', format='.0f'),
    color=alt.condition('datum.pct > 50', alt.value('white'), alt.value('black')),
)
heat + txt

alt.LayerChart(...)

## 9.3 — Component-level: digital vs humanities

The user hypothesis (and a common scholarly observation): in *Digital Humanities*, the **humanities** side is harder to translate than the **digital** side. "Digital" has settled into a stable cross-linguistic concept (computers, technology, electronic data); "humanities" is contested even within English-language academia (scholarly field? humanities-as-people? humanities-as-culture?).

Operationalise this by asking: when a rationale explicitly glosses one of the two components, *how clearly does it gloss each side*? A clear gloss of *digital* uses words like *technology*, *computational*, *electronic*, *computer-based*. A clear gloss of *humanities* uses words like *scholarship*, *humanistic*, *liberal arts*, *interdisciplinary*, *cultural studies*. We measure the rate at which rationales contain clear glosses for each side.

In [5]:
DIGITAL_GLOSS = [
    r'\bdigital\s+(?:technology|technologies|tools?|methods?|techniques?)\b',
    r'\bcomput(?:er|ational|ing)\b',
    r'\belectronic\b',
    r'\binformation\s+technology\b',
    r'\btechnolog(?:y|ies|ical)\b',
]

HUMANITIES_GLOSS_SCHOLARLY = [
    r'\b(?:liberal\s+arts|scholarly|scholarship|humanistic|humanist)\b',
    r'\b(?:cultural\s+studies|literary\s+studies|historical\s+studies)\b',
    r'\b(?:academic\s+field|academic\s+discipline)\b',
    r'\binterdisciplinary\s+(?:field|study|research|approach)\b',
]

HUMANITIES_GLOSS_GENERIC = [
    r'\bhumanities\s+(?:fields?|subjects?|disciplines?|areas?)\b',
    r'\bhuman(?:istic|ities)\s+(?:knowledge|topics|matters)\b',
    r'\bhuman\s+(?:culture|experience|society|history)\b',
]

def _any(rat, pats):
    return any(re.search(p, rat, re.IGNORECASE) for p in pats)

rationale_df['has_digital_gloss']            = rationale_df['rationale'].apply(lambda r: _any(r, DIGITAL_GLOSS))
rationale_df['has_humanities_gloss_scholar'] = rationale_df['rationale'].apply(lambda r: _any(r, HUMANITIES_GLOSS_SCHOLARLY))
rationale_df['has_humanities_gloss_generic'] = rationale_df['rationale'].apply(lambda r: _any(r, HUMANITIES_GLOSS_GENERIC))
rationale_df['has_humanities_gloss_any']     = rationale_df['has_humanities_gloss_scholar'] | rationale_df['has_humanities_gloss_generic']

component_summary = (
    rationale_df.groupby('service')[['has_digital_gloss',
                                      'has_humanities_gloss_scholar',
                                      'has_humanities_gloss_generic',
                                      'has_humanities_gloss_any']]
    .mean().mul(100).round(1)
)
print('Component-gloss prevalence (% of rationales):\n')
print(component_summary.to_string())
print('\nDigital-vs-humanities-any asymmetry (digital minus humanities, percentage points):')
print((component_summary['has_digital_gloss'] - component_summary['has_humanities_gloss_any']).round(1).to_string())

Component-gloss prevalence (% of rationales):

          has_digital_gloss  has_humanities_gloss_scholar  has_humanities_gloss_generic  has_humanities_gloss_any
service                                                                                                          
claude                 37.5                          40.4                          12.7                      46.5
deepseek               39.5                          38.1                          14.6                      45.5
gemini                 42.8                          44.0                          27.2                      57.1
gemma                  48.8                          16.2                          18.3                      31.2
llama                  34.6                          11.0                          14.5                      24.2
mistral                19.6                           7.1                           6.2                      12.5
openai                 56.0              

In [6]:
# Side-by-side bars: digital vs humanities gloss per service
comp_long = (component_summary[['has_digital_gloss', 'has_humanities_gloss_any']]
             .reset_index()
             .melt(id_vars='service', var_name='component', value_name='pct'))
comp_long['component'] = comp_long['component'].map({
    'has_digital_gloss': 'digital',
    'has_humanities_gloss_any': 'humanities (any gloss)',
})

alt.Chart(comp_long).mark_bar().encode(
    x=alt.X('pct:Q', title='% of rationales with clear gloss'),
    y=alt.Y('service:N', sort=svc_order, title=None),
    color=alt.Color('component:N',
                    scale=alt.Scale(domain=['digital', 'humanities (any gloss)'],
                                    range=['#1976d2', '#c0680c'])),
    yOffset='component:N',
    tooltip=['service:N','component:N','pct:Q'],
).properties(width=420, height=300,
             title=alt.Title('Component-gloss asymmetry across services',
                             subtitle='Larger gap = humanities is harder for that model to gloss clearly'))

alt.Chart(...)

## 9.4 — Variant effects on the definitional register

Does the `github_searcher` framing ("you are building a multilingual search corpus") shift the model's definitional register away from scholarly framing and toward something more pragmatic — corpus-building, search queries, topic tags? Does the `judge` variant aggregate scholarly framing from prior variants? This section cross-tabs register × variant within each service.

In [7]:
# Within-service variant effect on scholarly register
variant_register = (
    rationale_df.groupby(['service', 'variant'])[[f'reg_{r}' for r in REGISTER_PATTERNS]]
    .mean().mul(100).round(1)
    .rename(columns={f'reg_{r}': r for r in REGISTER_PATTERNS})
    .reset_index()
)

# Long format for plot
var_long = variant_register.melt(
    id_vars=['service','variant'],
    value_vars=['scholarly_methodological','human_centered','knowledge_generic'],
    var_name='register', value_name='pct',
)

alt.Chart(var_long).mark_bar().encode(
    x=alt.X('variant:N', sort=ENGLISH_VARIANTS, title=None),
    y=alt.Y('pct:Q', title='% of rationales'),
    color=alt.Color('register:N',
                    scale=alt.Scale(domain=['scholarly_methodological','human_centered','knowledge_generic'],
                                    range=['#1a6b30','#c0680c','#5a4a8c'])),
    column=alt.Column('service:N', sort=svc_order, title=None,
                       header=alt.Header(labelOrient='top', titleOrient='top')),
).properties(width=85, height=180,
             title=alt.Title('Variant effect on definitional register',
                             subtitle='Does github_searcher shift toward pragmatic framing? Does judge synthesise?'))

alt.Chart(...)

## 9.5 — Cross-reference with translation quality

Hypothesis: rationales that reach for scholarly framings produce translations that pass the quality-flag suite more often than rationales that stay literal-compositional or fall to generic-knowledge framings. Operationalise by joining each rationale row to its language's `flag_count` from `quality_flags.csv` and computing mean flag count by register.

**Note on causality**: this is associational, not causal. A scholarly rationale and a clean translation may both reflect that the model has stronger knowledge of the target language — the rationale isn't *causing* the cleaner translation; both are downstream of the same upstream model competence.

In [8]:
qf_path = os.path.join(DATA_DIR, 'translated_terms', TERM_SLUG, 'evaluation', 'quality_flags.csv')
qf = pd.read_csv(qf_path, dtype=str)
qf['flag_count'] = qf['flag_count'].astype(int)

rationale_q = rationale_df.merge(
    qf[['language_code', 'flag_count', 'language_family']],
    on='language_code', how='left',
)
rationale_q['flag_count'] = rationale_q['flag_count'].fillna(0).astype(int)

# Mean flag count by register
for register in REGISTER_PATTERNS:
    col = f'reg_{register}'
    a = rationale_q.loc[ rationale_q[col], 'flag_count'].mean()
    b = rationale_q.loc[~rationale_q[col], 'flag_count'].mean()
    print(f'  {register:30s}: mean flags  with={a:.2f}  without={b:.2f}  diff={a-b:+.2f}')

  compositional_literal         : mean flags  with=3.95  without=3.97  diff=-0.02
  scholarly_methodological      : mean flags  with=3.76  without=4.03  diff=-0.28
  human_centered                : mean flags  with=4.02  without=3.95  diff=+0.07
  knowledge_generic             : mean flags  with=4.16  without=3.91  diff=+0.25


In [9]:
# Per-service: mean flag count by scholarly-register status
svc_register_flags = (
    rationale_q.groupby(['service', 'reg_scholarly_methodological'])['flag_count']
    .mean().reset_index()
    .rename(columns={'reg_scholarly_methodological': 'has_scholarly'})
)
svc_register_flags['has_scholarly'] = svc_register_flags['has_scholarly'].map({True: 'scholarly', False: 'not scholarly'})

alt.Chart(svc_register_flags).mark_bar().encode(
    x=alt.X('flag_count:Q', title='mean flag count'),
    y=alt.Y('service:N', sort=svc_order, title=None),
    color=alt.Color('has_scholarly:N',
                    scale=alt.Scale(domain=['scholarly', 'not scholarly'],
                                    range=['#1a6b30', '#aaaaaa'])),
    yOffset='has_scholarly:N',
    tooltip=['service:N','has_scholarly:N','flag_count:Q'],
).properties(width=420, height=300,
             title=alt.Title('Mean flag count: scholarly-register vs not',
                             subtitle='If scholarly bars are shorter, richer rationales associate with cleaner translations'))

alt.Chart(...)

## 9.6 — Limitations and future LLM-as-judge validation

The keyword approach is *coarse*: it relies on closed lists of phrasal patterns that are inherently incomplete and biased toward English-language scholarly conventions of how DH is described. Several rationales that reach a scholarly register through different phrasing (e.g., *"intersection of the humanities and computational methods"*) may slip past the regex. Conversely, a rationale that mentions *"interdisciplinary"* about something unrelated to the field's identity would be miscounted as scholarly.

A natural validation step is **LLM-as-judge classification** on a stratified sample: take ~200 rationales sampled across services × variants × registers, present each to a separate LLM (Claude or GPT-4o) with a definitional-register classification schema, and compare the LLM's judgement to the keyword classifier's output. A high inter-rater agreement validates the keyword approach for paper-quality results; a low agreement flags specific register categories that need refinement.

**Other natural extensions:**

- **Family-level analysis**: does the model's definitional richness vary by *target* language family? Does Claude reach for scholarly framing more often when translating into Indo-European languages than into Niger-Congo languages?
- **Within-rationale shifts**: does the model start with a scholarly definition and then drift to a literal compositional one as it discusses target-language morphology? Sentence-level tagging would surface this.
- **Component asymmetry within a single sentence**: do the rationales that gloss *digital* also gloss *humanities*, or is the asymmetry happening *within* rationales (not across)?
- **Cross-reference with the success cases from §1.5**: in the languages where multiple models converge on the same translation (Māori `matihiko`, Corsican `umanità digitale`), do the rationales also converge on the same definitional register? If yes, this would strengthen the claim that scholarly definitions are a precondition for translation convergence.